# 03 - Naive Baselines (Global Mean & Popularity)

Two parameter-free baselines that define the performance floor every later model must
beat. Each is evaluated on the held-out test set and its results written to
`artifacts/metrics/all_metrics.json`.

In [1]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.evaluation.report import full_metrics, save_metric, top_n
from hybrid_recsys.config import RANDOM_STATE

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1100, height=550, scale=2)
    except Exception:
        pass
    fig.show()


EVAL_USERS, N_NEGATIVES = 1_000, 100
rng = np.random.default_rng(RANDOM_STATE)

movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)
all_movie_ids    = movies["movieId"].values
user_ratings_map = (
    train.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
)
eval_user_ids = rng.choice(
    test["userId"].unique(), size=min(EVAL_USERS, test["userId"].nunique()), replace=False
)
test_sample = test[test["userId"].isin(eval_user_ids)]
demo_user   = max(eval_user_ids, key=lambda u: len(user_ratings_map.get(u, {})))
print(f"Loaded splits - train {len(train):,}, test {len(test):,}; demo user = {int(demo_user)}")


Loaded splits - train 19,936,012, test 2,633,022; demo user = 127311


In [2]:
def ranking_curve(metrics, title):
    rows = [{"K": k, "Metric": m.capitalize(), "Value": metrics[f"k{k}"][m]}
            for k in [5, 10, 20] for m in ["precision", "recall", "f1"]]
    return px.line(pd.DataFrame(rows), x="K", y="Value", color="Metric", markers=True,
                   title=f"Ranking metrics @ K - {title}")


def error_hist(preds, title):
    err = test["rating"].to_numpy() - preds
    err = err[~np.isnan(err)]
    fig = px.histogram(err, nbins=50, title=f"Rating error (true - predicted) - {title}")
    fig.update_layout(showlegend=False, xaxis_title="true - predicted", bargap=0.02)
    return fig


def show_example(predict_fn, n=10, n_candidates=3000):
    seen = set(user_ratings_map.get(demo_user, {}))
    cand = rng.choice(all_movie_ids, size=min(n_candidates, len(all_movie_ids)), replace=False)
    recs = top_n(predict_fn, demo_user, seen, cand, movies, n=n)
    hist = (
        pd.DataFrame({"movieId": list(seen),
                      "rating": [user_ratings_map[demo_user][m] for m in seen]})
        .merge(movies[["movieId", "clean_title", "genres"]], on="movieId", how="left")
        .sort_values("rating", ascending=False).head(n)
    )
    return hist, recs


def star_graph(center, leaves, weights, title, name):
    k = len(leaves)
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    lx, ly = np.cos(angles), np.sin(angles)
    edge_x, edge_y = [], []
    for x, y in zip(lx, ly):
        edge_x += [0, x, None]
        edge_y += [0, y, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(color="#cccccc", width=1), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=lx, y=ly, mode="markers+text",
        marker=dict(size=16, color=list(weights), colorscale="Teal",
                    showscale=True, colorbar=dict(title="sim")),
        text=[f"{l}<br>{w:.2f}" for l, w in zip(leaves, weights)],
        textposition="top center", hoverinfo="text"))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text",
                             marker=dict(size=26, color="#EF553B"),
                             text=[center], textposition="bottom center", hoverinfo="text"))
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    save_fig(fig, name)
    return fig


## Global Mean — train + evaluate

In [3]:
global_mean = float(train["rating"].mean())
gm_predict = lambda u, i: global_mean

m, preds = full_metrics(gm_predict, test, test_sample, train_val, all_movie_ids,
                        n_negatives=N_NEGATIVES, random_state=RANDOM_STATE)
save_metric("Global Mean", m)
print(f"global mean = {global_mean:.3f}  |  RMSE={m['rmse']}  MAE={m['mae']}  F1@10={m['k10']['f1']}")

global mean = 3.549  |  RMSE=1.0609  MAE=0.8339  F1@10=0.0758


## Popularity — train + evaluate

In [4]:
item_pop = train.groupby("movieId").size().to_dict()
max_pop  = max(item_pop.values())

def pop_predict(u, i):
    return 0.5 + 4.5 * (item_pop.get(i, 0) / max_pop)

m, preds = full_metrics(pop_predict, test, test_sample, train_val, all_movie_ids,
                        n_negatives=N_NEGATIVES, random_state=RANDOM_STATE)
save_metric("Popularity", m)
print(f"RMSE={m['rmse']}  MAE={m['mae']}  F1@10={m['k10']['f1']}")

RMSE=2.712  MAE=2.4856  F1@10=0.609


### What Popularity recommends to everyone (top-10)

In [5]:
top_pop = (
    pd.Series(item_pop, name="ratings").sort_values(ascending=False).head(10)
    .rename_axis("movieId").reset_index()
    .merge(movies[["movieId", "clean_title", "genres"]], on="movieId")
)
display(top_pop[["clean_title", "genres", "ratings"]])

,clean_title,genres,ratings
0,"Shawshank Redemption, The",Crime|Drama,74930
1,Forrest Gump,Comedy|Drama|Romance|War,73311
2,Pulp Fiction,Comedy|Crime|Drama|Thriller,72221
3,"Silence of the Lambs, The",Crime|Horror|Thriller,66998
4,"Matrix, The",Action|Sci-Fi|Thriller,66510
5,Star Wars: Episode IV - A New Hope,Action|Adventure|Sci-Fi,62303
6,Jurassic Park,Action|Adventure|Sci-Fi|Thriller,57020
7,Schindler's List,Drama|War,53772
8,Braveheart,Action|Drama|War,53138
9,Fight Club,Action|Crime|Drama|Thriller,52616


### Popularity — ranking metrics @ K

In [6]:
save_fig(ranking_curve(m, "Popularity"), "eval_popularity_ranking")

## Takeaway

Global Mean anchors RMSE/MAE but cannot rank (constant output). Popularity ranks well under
sampled-negatives (relevant items skew popular) yet has meaningless RMSE. These are the bar
the real models must clear.